# 12.5 Geometric learning: from coordinates to invariant energies and equivariant forces

The molecular graphs in earlier parts describe connectivity. Atomistic models also use positions: changing a torsion changes the geometry while the covalent graph can stay the same. This lesson connects graph learning to the potential-energy surfaces, forces and vibrations introduced in Chapters 3 and 6–8.

## Learning objectives

1. Distinguish invariance from equivariance, and proper rotations from reflections.
2. Construct a radius graph and smooth distance features from a small molecular conformer.
3. Build a transparent scalar neural energy and differentiate it with respect to positions.
4. Test energy and force transformations, conservation identities, and a finite-difference derivative.
5. Explain chirality, cutoffs, periodic images, reference-state choices and the limits of pretrained potentials.

**Standalone, offline and short:** use the environment in [README.md](Readme.md). The only calculation is a small RDKit conformer and forward/gradient evaluations of a tiny CPU network. There is no training loop, downloaded model, GPU or earlier notebook output. The network has **random weights: its outputs are dimensionless illustrations, not chemical energies or validated forces**. Results go to `outputs/chapter12_part5/`.

**Before this lesson:** [12.4 Diagnostics and explanations](Chapter12_Part4.ipynb). For the physical context, revisit [molecular mechanics](Chapter02_Part2.ipynb), [DFT](Chapter06.ipynb), [vibrations](Chapter08_Part5.ipynb), and [molecular dynamics](Chapter08_Part7.ipynb).

### Start here: moving a molecule versus changing its shape

Rotating your view of a molecule changes its coordinate numbers but should preserve its energy. Bending a bond or turning a torsion changes the molecular shape and can change the energy. A force is an arrow: rotating the molecule should rotate that arrow too. These statements explain the words **invariant** (same scalar) and **equivariant** (correspondingly transformed vector).

**Research question:** could a connectivity-only model rank the conformers of one molecule? The butane example below shows why identical connectivity needs different 3D inputs for that task. Positions in a tensor have shape `(number of atoms, 3)`; row $i$ contains the $x,y,z$ coordinates of atom $i$. A distance removes the arbitrary origin and orientation but loses the sign of handedness.

**First pass:** conformer comparison → radius-graph picture → rotated force arrows → cutoff plot. **Deeper pass:** gradients, symmetry identities, and parity. The calculations test mathematical behavior of an untrained network. A research potential additionally needs reference energies/forces, units, chemical coverage, and validation on unseen configurations. Continue with [Part 6: real PyG data and layers](Chapter12_Part6.ipynb) for the library implementation sequence.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, rdMolDescriptors

OUT = Path("outputs/chapter12_part5")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 20261205
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DTYPE = torch.float64
print(f"RDKit {rdBase.rdkitVersion}; PyTorch {torch.__version__}; CPU, one thread")

## 12.5.1 Match the output to the symmetry

An **invariant** output stays the same after transforming the input. An **equivariant** output changes by the corresponding transformation. For column position vectors, write $\mathbf r'_i=Q\mathbf r_i+\mathbf t$, with $Q^\mathsf TQ=I$.

| Quantity for an isolated system | Expected transformation |
|---|---|
| Scalar potential energy $E$ | $E'=E$ |
| Force, a polar vector | $\mathbf F'_i=Q\mathbf F_i$; translation has no effect |
| A second-rank Cartesian tensor | $T'=QTQ^\mathsf T$ for this tensor convention |
| Atom-resolved outputs after relabeling atoms | Reorder with the atoms |
| Molecular scalar after consistent atom relabeling | Unchanged |

**SE(3)** consists of translations and proper rotations ($\det Q=+1$). **E(3)** also includes improper orthogonal transformations ($\det Q=-1$), such as a mirror reflection. Permuting atom indices is a separate symmetry: reorder positions and all associated atom features together. Swapping just coordinates between chemically different atoms changes the system.

These expectations depend on the target and environment. Rotating a molecule while keeping an external electric field fixed can change its energy; the environment must transform too for the isolated-system rule to apply. Polar vectors and axial vectors have different reflection parity. A dipole of a charged system also depends on the origin, so not every vector target is translation invariant. [EGNN symmetry formulation](https://proceedings.mlr.press/v139/satorras21a.html), [equivariant atomistic representations](https://arxiv.org/abs/2102.03150).

**Code convention:** positions are rows of an `(N, 3)` array, so rotation is `q @ Q.T` and vector outputs rotate the same way. We use dimensionless $\mathbf q=\mathbf r/\ell_0$ with $\ell_0=1\ \mathrm{Å}$. Thus the numbers initially equal the RDKit coordinates in Å, but the neural differentiation variable is dimensionless.

## 12.5.2 A reproducible chiral molecular geometry

We generate one neutral lactic-acid conformer from a stereochemically specified SMILES, with explicit hydrogens and ETKDGv3. ETKDG combines distance geometry with empirical conformational preferences. This is a generated geometry, not an experimental structure, a DFT calculation, a thermal ensemble or a proof of a global minimum. No geometry optimization is needed for the symmetry checks. [RDKit conformer-generation documentation](https://www.rdkit.org/docs/RDKit_Book.html#conformer-generation).

The atom table keeps the correspondence between element, graph index and coordinate. SMILES stereochemical tags constrain the embedding, but the neural model below receives only atomic numbers and coordinates. Charge, spin and electronic state would need an explicit policy or additional inputs in a physical model.

In [ ]:
SMILES = "C[C@H](O)C(=O)O"
mol = Chem.AddHs(Chem.MolFromSmiles(SMILES))
settings = AllChem.ETKDGv3()
settings.randomSeed = SEED
settings.numThreads = 1
settings.enforceChirality = True
if AllChem.EmbedMolecule(mol, settings) != 0:
    raise RuntimeError("The deterministic teaching conformer could not be embedded.")
coordinates_A = np.array(mol.GetConformer().GetPositions(), dtype=np.float64)
atomic_numbers = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.long)
symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
length_scale_A = 1.0
q = torch.tensor(coordinates_A / length_scale_A, dtype=DTYPE)
assert torch.isfinite(q).all() and q.shape == (mol.GetNumAtoms(), 3)
atom_table = pd.DataFrame({"atom_index": np.arange(len(q)), "element": symbols,
                           "atomic_number": atomic_numbers.numpy(),
                           "x_A": coordinates_A[:, 0], "y_A": coordinates_A[:, 1],
                           "z_A": coordinates_A[:, 2]})
display(atom_table)
print("Formula:", rdMolDescriptors.CalcMolFormula(mol))
print("Graph stereocenter labels:", Chem.FindMolChiralCenters(mol, includeUnassigned=True))
atom_table.to_csv(OUT / "generated_lactic_acid_coordinates.csv", index=False)

### Worked research representation check: two conformers, one connectivity

Imagine building a surrogate for a conformational energy scan from Chapter 2. We prescribe butane torsions of $60^\circ$ and $180^\circ$ using [RDKit's coordinate transformations](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolTransforms.html). The generated conformers are **not optimized minima**; they isolate the information supplied by geometry. Both have the same covalent graph and connectivity fingerprint, while their terminal carbon distance changes.

A deterministic model receiving only that connectivity and fixed atom/bond features must assign both the same prediction. It cannot learn a conformer-specific energy difference without extra information, however many training examples it receives. Coordinates or geometry-dependent features make this distinction available. A suitable research evaluation would hold out configurations or molecules according to the intended use, rather than distribute nearly identical trajectory frames randomly.

In [ ]:
from rdkit.Chem import rdMolTransforms, rdFingerprintGenerator
butane = Chem.AddHs(Chem.MolFromSmiles('CCCC'))
butane_settings = AllChem.ETKDGv3()
butane_settings.randomSeed = SEED
butane_settings.numThreads = 1
assert AllChem.EmbedMolecule(butane,butane_settings) == 0
conformer_copies = []
conformer_rows = []
fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
reference_bits = fingerprint_generator.GetFingerprint(butane).ToBitString()
fig = plt.figure(figsize=(10,4),layout='constrained')
for panel,angle in enumerate([60.,180.],start=1):
    candidate = Chem.Mol(butane)
    rdMolTransforms.SetDihedralDeg(candidate.GetConformer(),0,1,2,3,angle)
    xyz = np.asarray(candidate.GetConformer().GetPositions())[:4]
    distance = np.linalg.norm(xyz[3]-xyz[0])
    assert fingerprint_generator.GetFingerprint(candidate).ToBitString() == reference_bits
    np.testing.assert_array_equal(Chem.GetAdjacencyMatrix(candidate),Chem.GetAdjacencyMatrix(butane))
    conformer_copies.append(candidate)
    conformer_rows.append({'torsion_deg':angle,'terminal_C_distance_A':distance})
    ax = fig.add_subplot(1,2,panel,projection='3d')
    ax.plot(xyz[:,0],xyz[:,1],xyz[:,2],'o-',color='#31688e',lw=2,ms=8)
    ax.plot(xyz[[0,3],0],xyz[[0,3],1],xyz[[0,3],2],'--',color='#c6493d',lw=1)
    for atom,(x_coord,y_coord,z_coord) in enumerate(xyz):
        ax.text(x_coord,y_coord,z_coord+.15,f'C{atom}',fontsize=9)
    ax.set(xlabel='x / Å',ylabel='y / Å',zlabel='z / Å',
           title=f'Torsion {angle:.0f}°; C0–C3 = {distance:.2f} Å',
           xlim=(-2.5,2.5),ylim=(-2.5,2.5),zlim=(-2.5,2.5))
    ax.set_box_aspect((1,1,1))
    ax.view_init(elev=25,azim=30)
conformer_table = pd.DataFrame(conformer_rows)
assert abs(conformer_table.terminal_C_distance_A.diff().iloc[-1]) > .2
display(conformer_table)
conformer_table.to_csv(OUT / 'butane_conformer_representation.csv',index=False)
fig.savefig(OUT / 'same_graph_different_conformers.png',dpi=150)
plt.show()
print('Both conformers have identical connectivity fingerprints; H atoms omitted only from the plots.')

## 12.5.3 A radius graph is not a covalent-bond graph

For cutoff $r_c$, connect distinct atoms when their distance is below $r_c$. Nearby nonbonded atoms may therefore exchange messages. Each undirected pair is represented by **two directed edges**, one in each direction. We exclude the self pair $i=j$ in this isolated-molecule example.

We inspect all $N(N-1)$ candidate pairs because $N$ is tiny. This costs $O(N^2)$ and is not an efficient neighbor-list implementation for a large simulation. The graph is rebuilt at every forward evaluation so its edges follow the coordinates. A radius cutoff creates local interactions; it does not magically describe long-range electrostatics, dispersion or collective effects.

In [ ]:
CUTOFF = 2.8  # dimensionless q-distance; here equivalent to 2.8 Å

def radius_edges(positions, cutoff):
    count = len(positions)
    candidates = ~torch.eye(count, dtype=torch.bool, device=positions.device)
    receiver, sender = candidates.nonzero(as_tuple=True)
    distances = torch.linalg.vector_norm(positions[sender] - positions[receiver], dim=1)
    within_cutoff = distances < cutoff
    return receiver[within_cutoff], sender[within_cutoff], distances[within_cutoff]

receiver, sender, distances = radius_edges(q, CUTOFF)
assert torch.all(distances > 0) and torch.all(receiver != sender)
edge_pairs = set(zip(receiver.tolist(), sender.tolist()))
assert all((j, i) in edge_pairs for i, j in edge_pairs)
edges = pd.DataFrame({"receiver": receiver.numpy(), "sender": sender.numpy(),
                      "distance_over_length_scale": distances.numpy()})
print(f"{len(q)} atoms, {len(edges)} directed radius edges, {mol.GetNumBonds()} covalent bonds")
display(edges.head(8))
edges.to_csv(OUT / "radius_edges.csv", index=False)

# This is an x-y projection of the 3D geometry; edges were selected in 3D.
fig, ax = plt.subplots(figsize=(6.7, 4.5), layout="constrained")
for i, j in edge_pairs:
    if i < j:
        ax.plot(coordinates_A[[i, j], 0], coordinates_A[[i, j], 1], color="0.82", lw=0.8)
for bond in mol.GetBonds():
    pair = [bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]
    ax.plot(coordinates_A[pair, 0], coordinates_A[pair, 1], color="0.25", lw=1.8)
colors = [{"H": "#a9b1b5", "C": "#31688e", "O": "#c6493d"}[s] for s in symbols]
ax.scatter(coordinates_A[:, 0], coordinates_A[:, 1], c=colors, s=75, zorder=3)
for index, (x, y, _) in enumerate(coordinates_A):
    offset = {0: (-17, 5), 7: (5, -12)}.get(index, (4, 5))
    ax.annotate(f"{symbols[index]}{index}", (x, y), xytext=offset, textcoords="offset points", fontsize=8)
ax.set(xlabel="x / Å", ylabel="y / Å", aspect="equal",
       title="3D radius graph, projected onto x-y\nDark: covalent bonds; light: nearby pairs")
ax.margins(0.16)
fig.savefig(OUT / "radius_graph_projection.png", dpi=150)
plt.show()

## 12.5.4 Smooth radial features and the cutoff envelope

Distances are invariant to translation, rotation and reflection. We expand a dimensionless distance $d$ in eight Gaussian functions,

$$
g_k(d)=\exp[-\beta(d-\mu_k)^2],\qquad
f_c(d)=\begin{cases}\tfrac12[1+\cos(\pi d/r_c)],&d<r_c,\\0,&d\ge r_c.\end{cases}
$$

The centers $\mu_k$ span the cutoff interval; $\beta>0$ controls the widths. The envelope satisfies $f_c(r_c)=f'_c(r_c)=0$. Messages therefore vanish with a continuous first derivative when an edge disappears, provided the remaining operations are smooth and do not introduce cutoff-dependent normalization. The cosine envelope is generally **$C^1$ but not $C^2$** at the cutoff: its second derivative jumps. Continuous forces do not imply a smooth Hessian. Higher-order smooth envelopes matter for demanding vibration and molecular-dynamics applications.

Crucially, multiply the **final message filter** by the envelope. Sending already-enveloped features through an MLP with biases can produce a nonzero filter at the cutoff. Normalizing by the discrete neighbor count can also reintroduce jumps. We use an unnormalized sum. The use of learned distance filters is inspired by [SchNet](https://arxiv.org/abs/1706.08566); the small network here is an instructional simplification, not a reproduction of that architecture.

In [ ]:
def cutoff_envelope(distance, cutoff):
    interior = 0.5 * (1 + torch.cos(torch.pi * distance / cutoff))
    return torch.where(distance < cutoff, interior, torch.zeros_like(distance))

RADIAL_COUNT = 8
radial_centers = torch.linspace(0, CUTOFF, RADIAL_COUNT, dtype=DTYPE)
radial_beta = 1.0 / float(radial_centers[1] - radial_centers[0])**2

def gaussian_features(distance, centers, beta):
    return torch.exp(-beta * (distance[:, None] - centers[None, :])**2)

d_grid = torch.linspace(0, 1.2 * CUTOFF, 250, dtype=DTYPE)
envelope = cutoff_envelope(d_grid, CUTOFF)
radial_plot = gaussian_features(d_grid, radial_centers, radial_beta) * envelope[:, None]
fig, axes = plt.subplots(1, 2, figsize=(9.3, 3.5), layout="constrained")
axes[0].plot(d_grid.numpy(), radial_plot.numpy())
axes[0].set(xlabel="Dimensionless distance d", ylabel="Envelope × Gaussian feature", title="Eight radial channels")
axes[1].plot(d_grid.numpy(), envelope.numpy(), label="Cosine envelope")
axes[1].axvline(CUTOFF, linestyle=":", color="black", label="Cutoff")
axes[1].set(xlabel="Dimensionless distance d", ylabel="Envelope", title="A smooth first-derivative boundary")
axes[1].legend()
fig.savefig(OUT / "radial_features.png", dpi=150)
plt.show()

## 12.5.5 A small invariant scalar network

Let $h_i$ be an eight-channel embedding of atomic number $Z_i$. A learned radial filter supplies a message from atom $j$ to atom $i$:

$$
m_{ij}=f_c(d_{ij})\,\mathrm{MLP}_{r}(g(d_{ij}))\odot h_j,\quad
h'_i=h_i+\tanh\left(W\sum_{j\in\mathcal N(i)}m_{ij}+b\right),\quad
u=\sum_i(w^\mathsf T h'_i+b_u).
$$

Here $\odot$ means channel-wise multiplication. The sum over neighbors ignores their list order, and the sum over atoms ignores atom numbering. All geometric inputs are distances, so the output is E(3)-invariant. The hidden channels are scalar features, not Cartesian vector components. Differentiating the scalar output will nevertheless give an equivariant vector field.

An atomic sum is additive for separated, noninteracting components beyond the cutoff in this local architecture. It does not prove that a real charged system has no long-range interaction. Neither atomic contributions nor individual learned channels are unique physical observables. The model ignores bond orders, formal charge, spin and explicit chirality labels; this is enough to test geometry, not to specify a general electronic potential-energy surface.

In [ ]:
class TinyRadialEnergy(nn.Module):
    def __init__(self, cutoff=2.8, width=8, radial_count=8):
        super().__init__()
        self.cutoff = cutoff
        self.embedding = nn.Embedding(10, width)  # toy atom types: H, C, O (Z < 10)
        self.filter = nn.Sequential(nn.Linear(radial_count, width), nn.Tanh(), nn.Linear(width, width))
        self.update = nn.Linear(width, width)
        self.readout = nn.Linear(width, 1)
        centers = torch.linspace(0, cutoff, radial_count, dtype=DTYPE)
        self.register_buffer("centers", centers)
        self.beta = 1.0 / float(centers[1] - centers[0])**2

    def forward(self, numbers, positions):
        receiver, sender, distance = radius_edges(positions, self.cutoff)
        h = self.embedding(numbers)
        radial = gaussian_features(distance, self.centers, self.beta)
        # Apply the envelope AFTER the biased filter network.
        filters = self.filter(radial) * cutoff_envelope(distance, self.cutoff)[:, None]
        messages = filters * h[sender]
        aggregated = torch.zeros_like(h).index_add(0, receiver, messages)
        updated = h + torch.tanh(self.update(aggregated))
        atomic_outputs = self.readout(updated).squeeze(-1)
        # The zero term keeps a coordinate derivative defined even with no edges.
        return atomic_outputs.sum() + 0.0 * positions.sum()

torch.manual_seed(SEED)
model = TinyRadialEnergy(CUTOFF, width=8, radial_count=RADIAL_COUNT).to(dtype=DTYPE)
model.eval()
print("Random parameters:", sum(p.numel() for p in model.parameters()))
print("One scalar output u:", float(model(atomic_numbers, q).detach()))
print("No chemical energy or force data have been supplied to this network.")

## 12.5.6 Forces from a scalar derivative

For an actual potential energy, $\mathbf F_i=-\nabla_{\mathbf r_i}E$. Our dimensionless analogue is $\mathbf f_i=-\nabla_{\mathbf q_i}u$. If a trained model instead defined $E=E_0u$ and $\mathbf r=\ell_0\mathbf q$, then

$$
\mathbf F_i=\frac{E_0}{\ell_0}\mathbf f_i.
$$

For example, choosing an energy scale in kJ mol$^{-1}$ and length scale in Å gives molar force units kJ mol$^{-1}$ Å$^{-1}$. Per-molecule forces in newtons additionally require the energy-per-molecule conversion. **We assign no physical $E_0$ to these random weights.** The numbers and arrows below are dimensionless derivatives only.

`eval()` does not disable gradients. Do not wrap a force calculation in `no_grad()` or `inference_mode()`: we need autograd with respect to positions. For training on force errors, calculating these derivatives normally needs `create_graph=True` so that a force loss can subsequently differentiate with respect to parameters. We only evaluate and use its default `False`. [PyTorch autograd API](https://docs.pytorch.org/docs/2.11/generated/torch.autograd.grad.html).

Translation-invariant energy implies $\sum_i\mathbf f_i=0$, and rotation-invariant energy implies $\sum_i\mathbf q_i\times\mathbf f_i=0$. These are net-force and net-torque identities for the isolated system. A differentiable scalar also defines a conservative force field. Symmetry alone for a directly predicted vector field would not guarantee conservativity, and numerical time integration can still produce energy drift even for a conservative model.

In [ ]:
def energy_and_forces(numbers, positions):
    coordinates = positions.detach().clone().requires_grad_(True)
    energy = model(numbers, coordinates)
    forces, = torch.autograd.grad(energy, coordinates)
    return energy.detach(), -forces.detach()

u_reference, f_reference = energy_and_forces(atomic_numbers, q)
net_force = f_reference.sum(dim=0)
net_torque = torch.linalg.cross(q - q.mean(dim=0), f_reference, dim=1).sum(dim=0)
assert torch.isfinite(f_reference).all()
torch.testing.assert_close(net_force, torch.zeros(3, dtype=DTYPE), rtol=0, atol=1e-11)
torch.testing.assert_close(net_torque, torch.zeros(3, dtype=DTYPE), rtol=0, atol=1e-11)
force_table = atom_table[["atom_index", "element"]].copy()
force_table[["f_x_dimensionless", "f_y_dimensionless", "f_z_dimensionless"]] = f_reference.numpy()
display(force_table)
print("Net force:", net_force.numpy())
print("Net torque:", net_torque.numpy())
force_table.to_csv(OUT / "dimensionless_forces.csv", index=False)

## 12.5.7 Test transformations rather than relying on the model name

The checks use double precision and absolute tolerances, not exact bitwise equality. Relabeling atoms can change the order of floating-point additions. We test a proper 3D rotation, a translation and an atom permutation separately. The force comparison follows the corresponding transformation rather than incorrectly demanding unchanged Cartesian components.

The algebra of this architecture explains the symmetry for arbitrary valid coordinates. A few numerical examples are useful implementation checks, not a general proof for a different architecture. Atomic collisions make the derivative of a distance singular; the generated conformer has distinct coordinates, and the finite-difference check will stay away from edge boundaries.

In [ ]:
az, ay = 0.7, -0.45
Rz = torch.tensor([[np.cos(az), -np.sin(az), 0], [np.sin(az), np.cos(az), 0], [0, 0, 1]], dtype=DTYPE)
Ry = torch.tensor([[np.cos(ay), 0, np.sin(ay)], [0, 1, 0], [-np.sin(ay), 0, np.cos(ay)]], dtype=DTYPE)
rotation = Rz @ Ry
torch.testing.assert_close(rotation.T @ rotation, torch.eye(3, dtype=DTYPE))
assert np.isclose(torch.linalg.det(rotation).item(), 1.0)
translation = torch.tensor([1.1, -0.8, 0.4], dtype=DTYPE)
permutation = torch.tensor(np.random.default_rng(SEED).permutation(len(q)), dtype=torch.long)
test_cases = [
    ("translation", atomic_numbers, q + translation, f_reference),
    ("proper rotation", atomic_numbers, q @ rotation.T, f_reference @ rotation.T),
    ("atom permutation", atomic_numbers[permutation], q[permutation], f_reference[permutation]),
]
symmetry_records = []
for name, numbers, positions, expected_force in test_cases:
    output, force = energy_and_forces(numbers, positions)
    torch.testing.assert_close(output, u_reference, rtol=0, atol=1e-11)
    torch.testing.assert_close(force, expected_force, rtol=0, atol=1e-11)
    symmetry_records.append({"transformation": name,
                             "absolute_energy_error": abs(float(output - u_reference)),
                             "maximum_force_component_error": float((force - expected_force).abs().max())})
display(pd.DataFrame(symmetry_records))

In [ ]:
# Force arrows share a single display multiplier; only their x-y projection is shown.
rotated_q = q @ rotation.T
_, rotated_force = energy_and_forces(atomic_numbers, rotated_q)
arrow_multiplier = 0.7 / float(torch.linalg.vector_norm(f_reference, dim=1).max())
display_extent = torch.cat([q, rotated_q, q + arrow_multiplier * f_reference,
                           rotated_q + arrow_multiplier * rotated_force])
plot_bound = float(display_extent[:, :2].abs().max()) + 0.35
fig, axes = plt.subplots(1, 2, figsize=(10, 4.1), layout="constrained")
for ax, positions, forces, title in [
    (axes[0], q, f_reference, "Original coordinates"),
    (axes[1], rotated_q, rotated_force, "Same geometry after a 3D rotation"),
]:
    array, arrows = positions.numpy(), forces.numpy() * arrow_multiplier
    for bond in mol.GetBonds():
        pair = [bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]
        ax.plot(array[pair, 0], array[pair, 1], color="0.7", lw=1)
    ax.scatter(array[:, 0], array[:, 1], c=colors, s=45, zorder=3)
    ax.quiver(array[:, 0], array[:, 1], arrows[:, 0], arrows[:, 1], angles="xy",
              scale_units="xy", scale=1, width=0.006, color="#7543a6")
    ax.set(xlabel="Dimensionless q_x", ylabel="Dimensionless q_y", title=title, aspect="equal",
           xlim=(-plot_bound, plot_bound), ylim=(-plot_bound, plot_bound))
fig.suptitle("x-y projections of coordinates and dimensionless gradient forces\nRandom neural model; common arrow multiplier")
fig.savefig(OUT / "force_rotation_projection.png", dpi=150)
plt.show()

### Independent derivative check

For a fixed unit direction $v$ in the $3N$-dimensional coordinate space, compare

$$
\frac{u(q+hv)-u(q-hv)}{2h}\quad\text{with}\quad
\nabla_q u\mathbin{:}v=-\sum_i\mathbf f_i\cdot\mathbf v_i.
$$

The central difference uses two forward evaluations instead of autograd. Its truncation error typically falls as $h^2$ for a smooth function until numerical cancellation dominates. We check several small steps, without moving any pair across the cutoff. This tests the sign and differentiation of the implemented function; it does not validate that function against quantum chemistry.

In [ ]:
direction = torch.tensor(np.random.default_rng(SEED + 1).normal(size=q.shape), dtype=DTYPE)
direction /= torch.linalg.vector_norm(direction)
analytic_directional_derivative = float(-(f_reference * direction).sum())
all_pair_distances = torch.pdist(q)
cutoff_margin = float(torch.min(torch.abs(all_pair_distances - CUTOFF)))
steps = [1e-3, 1e-4, 1e-5]
assert cutoff_margin > 2 * max(steps), "Choose smaller steps to avoid a graph boundary."
assert float(all_pair_distances.min()) > 2 * max(steps)
derivative_records = []
for step in steps:
    with torch.no_grad():
        plus = model(atomic_numbers, q + step * direction)
        minus = model(atomic_numbers, q - step * direction)
        finite_difference = float((plus - minus) / (2 * step))
    derivative_records.append({"step": step, "finite_difference": finite_difference,
                               "autograd": analytic_directional_derivative,
                               "absolute_error": abs(finite_difference - analytic_directional_derivative)})
derivative_check = pd.DataFrame(derivative_records)
display(derivative_check)
assert derivative_check.iloc[-1]["absolute_error"] < 1e-7
derivative_check.to_csv(OUT / "finite_difference_check.csv", index=False)

## 12.5.8 Reflection, chirality and the target's parity

Reflecting every coordinate preserves all pairwise distances. A model using only atomic numbers and distances therefore assigns the same scalar to mirror geometries. That is appropriate for the energy of isolated enantiomers under a parity-conserving, field-free molecular Hamiltonian (the usual nonrelativistic chemical approximation). It is insufficient for a **pseudoscalar** response that changes sign under reflection, or for distinguishing interactions with a fixed chiral environment.

A distance-based invariant scalar cannot distinguish enantiomers by itself. Full pairwise distances specify a generic labeled geometry only up to a rigid isometry (translation and an orthogonal transformation, including reflection); a truncated radius graph loses additional information. An E(3)-equivariant architecture can still carry odd-parity features and predict a pseudoscalar if its output transformation is chosen accordingly. E(3) equivariance does not require every possible output to be reflection-even.

For three **identity-ordered** neighbor vectors at the tetrahedral center, the scalar triple product $\chi=\det[v_1;v_2;v_3]$ keeps its sign under proper rotation and changes sign under reflection. This is a geometric handedness check, **not an R/S assignment**: exchanging the neighbor order also changes the sign. We reflect coordinate arrays only; we do not silently change the original RDKit molecule's stored stereochemical tags.

In [ ]:
reflection = torch.diag(torch.tensor([-1.0, 1.0, 1.0], dtype=DTYPE))
mirror_q = q @ reflection.T
mirror_u, mirror_f = energy_and_forces(atomic_numbers, mirror_q)
torch.testing.assert_close(mirror_u, u_reference, rtol=0, atol=1e-11)
torch.testing.assert_close(mirror_f, f_reference @ reflection.T, rtol=0, atol=1e-11)
torch.testing.assert_close(torch.pdist(mirror_q), torch.pdist(q), rtol=0, atol=1e-12)
center = Chem.FindMolChiralCenters(mol, includeUnassigned=False)[0][0]
ordered_neighbors = sorted(a.GetIdx() for a in mol.GetAtomWithIdx(center).GetNeighbors()
                           if a.GetAtomicNum() > 1)
assert len(ordered_neighbors) == 3

def signed_volume(positions):
    return torch.linalg.det(positions[ordered_neighbors] - positions[center])

chi = signed_volume(q)
assert abs(float(chi)) > 1e-3
torch.testing.assert_close(signed_volume(q @ rotation.T), chi)
torch.testing.assert_close(signed_volume(mirror_q), -chi)
display(pd.DataFrame({"geometry": ["original", "proper rotation", "mirror reflection"],
                      "identity_ordered_triple_product": [float(chi), float(signed_volume(q @ rotation.T)),
                                                          float(signed_volume(mirror_q))]}))
symmetry_records.append({"transformation": "reflection",
                         "absolute_energy_error": abs(float(mirror_u - u_reference)),
                         "maximum_force_component_error": float((mirror_f - f_reference @ reflection.T).abs().max())})
print("Mirror energy matches, while the geometric handedness changes sign.")

## 12.5.9 What happens when an edge disappears?

Scan a two-atom toy system through the cutoff using the same network. Plot its interaction contribution $u(d)-u(d_{\mathrm{far}})$ and the dimensionless radial force on the right-hand atom. Beyond the cutoff, the atoms retain their individual readout contributions but no message connects them. The envelope makes both the interaction and its first derivative approach zero; an abrupt unweighted edge deletion would generally cause a jump.

The neighbor-list membership comparison is discrete and is not differentiated by autograd. Smooth limiting behavior must come from the architecture. A capped neighbor list, changing nearest-neighbor order, discontinuous normalization, nonsmooth activations or inadequate cutoff envelope can defeat that behavior. Even with this cosine envelope, the second derivative generally jumps at $r_c$; do not assume a smoothly varying vibrational Hessian. Practical interatomic potentials require explicit smoothness and stability checks. [Smooth interatomic-potential design](https://arxiv.org/abs/2502.12147).

In [ ]:
two_numbers = torch.tensor([6, 8], dtype=torch.long)  # illustrative atom types, not a CO potential
scan_distances = np.linspace(CUTOFF - 0.35, CUTOFF + 0.35, 61)
scan_values = []
for distance in scan_distances:
    two_q = torch.tensor([[0., 0., 0.], [distance, 0., 0.]], dtype=DTYPE)
    output, force = energy_and_forces(two_numbers, two_q)
    scan_values.append([distance, float(output), float(force[1, 0])])
scan = pd.DataFrame(scan_values, columns=["dimensionless_distance", "u", "right_atom_radial_force"])
scan["interaction_u"] = scan["u"] - scan.iloc[-1]["u"]
assert np.max(np.abs(scan.loc[scan.dimensionless_distance > CUTOFF, "right_atom_radial_force"])) < 1e-12
fig, axes = plt.subplots(1, 2, figsize=(9.3, 3.5), layout="constrained")
for ax, column, ylabel in zip(axes, ["interaction_u", "right_atom_radial_force"],
                            ["Dimensionless interaction u", "Dimensionless radial force"]):
    ax.plot(scan.dimensionless_distance, scan[column])
    ax.axvline(CUTOFF, color="black", linestyle=":", label="Cutoff")
    ax.set(xlabel="Dimensionless distance d", ylabel=ylabel)
    ax.legend()
fig.suptitle("Two-atom cutoff check: random model, not a chemical bond curve")
fig.savefig(OUT / "cutoff_scan.png", dpi=150)
plt.show()
scan.to_csv(OUT / "cutoff_scan.csv", index=False)

## 12.5.10 From this toy model to chemical potentials

### Decide what physical function is being approximated

A learned potential approximates a reference surface with specified composition, total charge, spin/electronic state, method and environment. DFT labels at different functionals or settings do not automatically share one consistent energy surface. Force labels must use the same convention and units as the energy derivative. An energy/force training loss might combine squared energy and force errors with declared weights; those terms have different units and numbers of components. Total-energy offsets and atomic reference energies also need a defined convention.

Forces contain local slope information, but fitting many components from nearby trajectory frames does not create independent chemical coverage. Training and test splits should separate the intended generalization units: molecules, conformers, trajectory segments, temperatures, chemical reactions or materials families. A random frame split can give deceptively small errors. Include strained structures and relevant reactive configurations when the application needs them, and test against independent reference calculations. Symmetry checks, low test MAE and short stable trajectories answer different questions; none alone establishes safe long-time dynamics.

### 2D graphs, conformers and ensembles

- A 2D bond graph may be appropriate for a connectivity-based property model; it does not specify torsions, intermolecular arrangements or a crystal packing.
- One 3D conformer is one configuration. Thermodynamic observables may require an ensemble and a stated temperature, solvent and statistical weighting.
- A local sum with a short radius cutoff is incomplete for some long-range effects unless the model adds suitable electrostatics, dispersion, global information or another justified treatment.
- The random model here has no repulsive-core guarantee, calibrated uncertainty or physical minimum. We do not run dynamics or geometry optimization with it.

### Periodic materials need lattice images

With cell vectors stored as the **rows** of matrix $A$, a periodic neighbor displacement is

$$
\mathbf d_{ij\mathbf n}=\mathbf r_j+\mathbf n A-\mathbf r_i,\qquad \mathbf n\in\mathbb Z^3.
$$

An edge must retain its lattice shift $\mathbf n$, not just atom indices. Self images with $i=j$ and $\mathbf n\ne0$ can be neighbors. A radius can contain several images of the same atom. A simple minimum-image rule is insufficient for every cutoff or skewed cell; use a validated periodic neighbor implementation. Rotating a periodic system means rotating its lattice too. Stress additionally involves a cell/strain derivative and a stated sign and volume convention; it is not merely another name for atomic force. [ASE periodic-neighbor API](https://wiki.fysik.dtu.dk/ase/ase/neighborlist.html) describes an established implementation; ASE is not needed to execute this notebook.

## 12.5.11 Representative architecture families and the current direction

The table is a reading map, not a ranking. Architecture, training data, target conventions and cost all affect the result. A name containing “equivariant” does not specify the complete model's reflection parity, smoothness, charge handling or applicability.

| Family and primary source | Main idea | Question to examine |
|---|---|---|
| [SchNet](https://arxiv.org/abs/1706.08566) | Continuous filters generated from distances; scalar atom features | Is the cutoff and readout appropriate for the target? |
| [DimeNet](https://arxiv.org/abs/2003.03123) | Directional edge messages use distances and angles | What angular information is retained, and what is its computational cost? |
| [PaiNN](https://arxiv.org/abs/2102.03150) | Coupled scalar and vector channels | How do vector or tensor properties transform? |
| [EGNN](https://proceedings.mlr.press/v139/satorras21a.html) | E(n)-equivariant scalar/coordinate updates using relative geometry | Is a coordinate update being interpreted correctly, rather than assumed to be a physical force? |
| [NequIP](https://arxiv.org/abs/2101.03164), [MACE](https://arxiv.org/abs/2206.07697) | Equivariant features and richer many-body interactions for interatomic potentials | Which reference surface, elements, states and configurations were learned? |
| [EquiformerV2](https://arxiv.org/abs/2306.12059) | Attention with geometric, higher-degree equivariant representations | Are positional encodings and every output operation compatible with the required symmetry? |

Ordinary graph attention does not by itself enforce 3D equivariance. Cartesian coordinates inserted into an unconstrained MLP or generic transformer can break rotational symmetry. Conversely, equivariant layers can be combined with attention when their features and operations transform consistently. Attention weights are not automatically causal explanations.

Broader pretrained atomistic models, including [MACE-MP](https://arxiv.org/abs/2401.00096) and [UMA](https://arxiv.org/abs/2506.23971), extend the scale and diversity of training and can provide starting points for domain-specific work. “Foundation” or “universal” does not mean exact quantum mechanics or unrestricted chemical coverage. Before choosing a checkpoint, inspect its precise version, license, training-domain overlap, elements, charge/spin support, reference method, units and energy/force consistency. Benchmark datasets such as [OMol25](https://arxiv.org/abs/2505.08762) also make the reference electronic-structure method part of the learning problem. A materials-trained potential and a molecular potential need not share the same energy convention or domain. No pretrained model is required here.

In [ ]:
symmetry_table = pd.DataFrame(symmetry_records)
symmetry_table.to_csv(OUT / "symmetry_checks.csv", index=False)
record = {
    "purpose": "untrained dimensionless geometric-ML demonstration; not a physical potential",
    "smiles": SMILES, "geometry": "RDKit ETKDGv3, explicit H, enforceChirality=True; no optimization",
    "seed": SEED, "rdkit_version": rdBase.rdkitVersion, "torch_version": str(torch.__version__),
    "coordinates_input_unit": "angstrom", "length_scale_angstrom": length_scale_A,
    "neural_coordinate_and_output_units": "dimensionless q and u; f=-du/dq",
    "cutoff_dimensionless": CUTOFF, "radial_channels": RADIAL_COUNT, "dtype": "float64",
    "model": "one scalar distance-filter message layer, width 8; random weights",
    "parameter_count": sum(p.numel() for p in model.parameters()),
    "maximum_energy_symmetry_error": float(symmetry_table.absolute_energy_error.max()),
    "maximum_force_symmetry_error": float(symmetry_table.maximum_force_component_error.max()),
    "smallest_step_finite_difference_error": float(derivative_check.iloc[-1].absolute_error),
}
(OUT / "experiment.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
torch.save({"state_dict": model.state_dict(), "record": record}, OUT / "random_demo_weights.pt")
print("Saved generated coordinates, graph, derivative/symmetry checks, random weights, figures and metadata.")

## Exercises

1. Why should rotating a molecule change force components even when its scalar energy does not change? Write the operation for row-vector arrays.
2. Does an invariant energy guarantee zero force on every atom? Which two sums should vanish for this isolated system?
3. If $E_0=2$ kJ $mol^{-1}$ and $\ell_0=0.5$ Å, what physical force component corresponds to $f_x=0.3$? Are those scales assigned to the random network in this notebook?
4. Why can an MLP applied after an envelope produce a discontinuity at the cutoff? Why can dividing by the number of neighbors do the same?
5. A distance-only energy is equal for the two reflected lactic-acid geometries. Does that mean the molecule is achiral? Could such a scalar directly represent an optical-rotation pseudoscalar?
6. Why does a conservative neural force field still require a molecular-dynamics time-step and stability check? Is a finite-difference agreement a test of chemical accuracy?
7. What is missing if a periodic edge stores only `(i, j)`? Why can simply discarding all `i == j` edges be wrong in a small periodic cell?
8. A pretrained model gives a low energy MAE on random frames from a trajectory. Propose three further checks before using it on a different charge state or reaction pathway.

### Worked answers

1. Force is a vector tied to the geometry. With row arrays use `q_rotated = q @ Q.T` and `f_rotated = f @ Q.T`; the scalar energy stays unchanged under a proper rotation of the complete isolated system.
2. No. Individual forces can be nonzero. Translation and rotation invariance imply zero net force $\sum_i f_i$ and net torque $\sum_i q_i\times f_i$, respectively.
3. $(E_0/\ell_0)f_x=1.2$ kJ $mol^{-1}$ $Å^{-1}$. These are hypothetical scales, not a physical interpretation assigned to our random weights.
4. A biased MLP need not map a zero feature vector to zero. Multiply its final filter by an envelope with the needed boundary derivatives. A discrete change in neighbor count can change every normalized message abruptly.
5. No. Reflection preserves distances but reverses handedness. The even scalar cannot express a sign-changing pseudoscalar without a representation/output that carries the required odd parity.
6. Discrete integration has numerical error, and the learned surface may have unphysical regions. Finite differences check derivatives of the implemented function, not agreement with a reference potential.
7. The lattice-image shift is missing. Another periodic image of the same atom can be a genuine neighbor, even though its index equals the receiver's index.
8. Check the checkpoint's supported charge/spin states and reference method; separate related frames/chemical groups in evaluation; compare new charged/reactive configurations and forces with independent quantum calculations. Also examine short-time stability, cutoff smoothness, relevant long-range physics and uncertainty outside the training domain.

**Takeaway:** geometric symmetry is a useful architectural constraint and a testable property. It complements clear targets, suitable chemical coverage and careful evaluation; it does not replace them.

[Back to 12.4](Chapter12_Part4.ipynb) · [Start Chapter 12](Chapter12_Part1.ipynb) · [Course index](Readme.md)